In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.besstie_data_loader import get_BESSTIE_splits
from src.eda_distributions import EDA, get_sarcasm_extremes, variety_slang

In [ ]:
df_all, df_train, df_val,df_test = get_BESSTIE_splits()

eda = EDA(df_all, df_train, df_val, df_test)

eda.sarcasm_dist()


In [ ]:
eda.sentiment_dist()

In [ ]:
variety_counts, source_counts = eda.variety_source_dist(eda.df_all)

In [ ]:
split_distribution = eda.split_distribution_per_variety()
print(split_distribution)

In [ ]:
source_variety_crosstab = eda.source_per_variety()

In [ ]:
sarcasm_source_crosstab = eda.sarcasm_by_source()

In [ ]:
sentiment_source_crosstab = eda.sentiment_by_source()

In [ ]:
sarcasm_sentiment_crosstab = eda.sarcasm_sentiment_correlation()

In [ ]:
overall_sarcasm, per_variety_sarcasm, per_split_sarcasm = eda.sarcasm_imbalance()

In [ ]:
overall_sentiment, per_variety_sentiment, per_split_sentiment = eda.sentiment_imbalance()

# Overall Sentiment
print("\n OVERALL SENTIMENT DISTRIBUTION:")
print(f"   Positive (1): {overall_sentiment[1]:.2f}%")
print(f"   Negative (0): {overall_sentiment[0]:.2f}%")

#  Per Variety Sentiment
print("\n SENTIMENT BY VARIETY (%):")
print(f"{'Variety':<12} {'Positive (1)':<15} {'Negative (0)':<15}")
for variety in per_variety_sentiment.index:
    pos_pct = per_variety_sentiment.loc[variety, 1]
    neg_pct = per_variety_sentiment.loc[variety, 0]
    print(f"{variety:<12} {pos_pct:.2f}%{' ':<8} {neg_pct:.2f}%")

# Per Variety Per Split Sentiment
print("\n SENTIMENT BY VARIETY & SPLIT (%):")
print(f"{'Variety':<12} {'Split':<12} {'Positive (1)':<15} {'Negative (0)':<15}")

for (variety, split) in per_split_sentiment.index:
    pos_pct = per_split_sentiment.loc[(variety, split), 1]
    neg_pct = per_split_sentiment.loc[(variety, split), 0]
    print(f"{variety:<12} {split:<12} {pos_pct:.2f}%{' ':<8} {neg_pct:.2f}%")

In [ ]:
variety_length, sentiment_length, sarcasm_length = eda.text_length(save=True)

In [ ]:
domain_length, google_vocab, reddit_vocab, overlap = eda.compare_domains(save=True)

In [ ]:
pos_data, sarc_pos, non_sarc_pos = eda.pos_for_sarcasm()
print(f"{'POS Tag':<12} {'Sarcastic %':<15} {'Non-sarcastic %':<15}")
for pos, sarc_pct, non_pct in zip(pos_data['pos_tags'], pos_data['sarcastic_pcts'], pos_data['non_sarcastic_pcts']):
    print(f"{pos:<12} {sarc_pct:.1f}%{' ':<10} {non_pct:.1f}%")

In [ ]:
slang_results = variety_slang(df_all)
for variety, slang_found in slang_results.items():
    print(f"\n📌 {variety.upper()}:")
    print(f"   Found {len(slang_found)} slang terms")

    if slang_found:
        print("   Examples:")
        for slang, example in slang_found[:3]:
            example_short = example[:100] + "..." if len(example) > 100 else example
            print(f"      • '{slang}': {example_short}")

In [ ]:
sarcasm_patterns, examples = eda.sarcastic_phrases_analysis()

In [ ]:
sarcasm_extremes = get_sarcasm_extremes(per_variety_sarcasm)
print("SARcASM EXTREMES")
print(f"Most sarcastic variety: {sarcasm_extremes['most_sarcastic']} ({sarcasm_extremes['most_sarcastic_pct']:.2f}%)")
print(f"Least sarcastic variety: {sarcasm_extremes['least_sarcastic']} ({sarcasm_extremes['least_sarcastic_pct']:.2f}%)")

In [ ]:
from sklearn.linear_model import LogisticRegression
import joblib
import os


def train_logistic_regression(X_train, y_train, task_name="sentiment", save_path="./models"):
    os.makedirs(save_path, exist_ok=True)

    LR_model = LogisticRegression(
        C=1.0,
        max_iter=1000,
        class_weight='balanced',
        random_state=42,
        solver='liblinear'
    )

    # Train
    LR_model.fit(X_train, y_train)

    # Save model
    joblib.dump(LR_model, f"{save_path}/logreg_{task_name}.pkl")

    return model


def train_both_models(X_train, y_train_sent, y_train_sarc, save_path="./models"):

    # sentiment model
    Sentiment_model = train_logistic_regression(
        X_train, y_train_sent,
        task_name="Sentiment",
        save_path=save_path
    )

    # sarcasm model
    Sarcasm_model = train_logistic_regression(
        X_train, y_train_sarc,
        task_name="Sarcasm",
        save_path=save_path
    )

    return Sentiment_model, Sarcasm_model